In [2]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
DATASET_PATH = "../dataset/UNSW_NB15_testing-set.parquet"

dataset = pd.read_parquet(DATASET_PATH)

In [4]:
LIVE_FEATURES = [
    "dur",
    "proto",
    "spkts",
    "dpkts",
    "sbytes",
    "dbytes",
    "rate",
    "sload",
    "dload",
    "sinpkt",
    "dinpkt",
    "smean",
    "dmean"
]

dataset[LIVE_FEATURES].describe().T

,count,mean,std,min,25%,50%,75%,max
dur,82332.0,1.006756e+00,4.710444e+00,0.0,0.000008,0.014138,7.193602e-01,5.999999e+01
spkts,82332.0,1.866647e+01,1.339164e+02,1.0,2.000000,6.000000,1.200000e+01,1.064600e+04
dpkts,82332.0,1.754594e+01,1.155741e+02,0.0,0.000000,2.000000,1.000000e+01,1.101800e+04
sbytes,82332.0,7.993908e+03,1.716423e+05,24.0,114.000000,534.000000,1.280000e+03,1.435577e+07
dbytes,82332.0,1.323379e+04,1.514715e+05,0.0,0.000000,178.000000,9.560000e+02,1.465753e+07
rate,82332.0,8.241089e+04,1.486204e+05,0.0,28.606114,2650.176758,1.111111e+05,1.000000e+06
sload,82332.0,6.454902e+07,1.798618e+08,0.0,11202.466797,577003.218750,6.514286e+07,5.268000e+09
dload,82332.0,6.305470e+05,2.393000e+06,0.0,0.000000,2112.951416,1.585808e+04,2.082111e+07
sinpkt,82332.0,7.553943e+02,6.182616e+03,0.0,0.008000,0.557929,6.340944e+01,6.000999e+04
dinpkt,82332.0,1.217013e+02,1.292378e+03,0.0,0.000000,0.010000,6.313637e+01,5.773924e+04


In [5]:
normal_data = dataset[dataset["label"] == 0]

normal_data[LIVE_FEATURES].describe().T

,count,mean,std,min,25%,50%,75%,max
dur,37000.0,1.012727e+00,4.063714e+00,0.0,0.002063,0.222841,0.865563,5.999999e+01
spkts,37000.0,2.277697e+01,4.610088e+01,1.0,4.000000,10.000000,16.000000,6.900000e+02
dpkts,37000.0,2.541532e+01,8.272737e+01,0.0,2.000000,8.000000,18.000000,1.432000e+03
sbytes,37000.0,4.072377e+03,1.495298e+04,46.0,520.000000,974.000000,2334.000000,3.391000e+05
dbytes,37000.0,1.870493e+04,1.090385e+05,0.0,164.000000,354.000000,1890.000000,1.925422e+06
rate,37000.0,2.834999e+04,1.029019e+05,0.0,22.889781,118.625534,3125.000000,1.000000e+06
sload,37000.0,3.975339e+07,1.908562e+08,0.0,8326.597900,91269.683594,837177.281250,4.368000e+09
dload,37000.0,1.373613e+06,3.423036e+06,0.0,1822.659058,7679.021729,657433.062500,2.082111e+07
sinpkt,37000.0,1.581859e+03,9.112200e+03,0.0,0.300585,15.836928,81.971752,6.000999e+04
dinpkt,37000.0,1.754665e+02,1.713021e+03,0.0,0.008000,1.209529,79.794983,5.773924e+04


In [6]:
attack_data = dataset[dataset["label"] == 1]

attack_data[LIVE_FEATURES].describe().T

,count,mean,std,min,25%,50%,75%,max
dur,45332.0,1.001882e+00,5.178826e+00,0.0,0.000006,1.000000e-05,5.603355e-01,5.999953e+01
spkts,45332.0,1.531148e+01,1.755324e+02,1.0,2.000000,2.000000e+00,1.000000e+01,1.064600e+04
dpkts,45332.0,1.112294e+01,1.363165e+02,0.0,0.000000,0.000000e+00,8.000000e+00,1.101800e+04
sbytes,45332.0,1.119466e+04,2.308732e+05,24.0,114.000000,2.000000e+02,8.220000e+02,1.435577e+07
dbytes,45332.0,8.768235e+03,1.786677e+05,0.0,0.000000,0.000000e+00,3.540000e+02,1.465753e+07
rate,45332.0,1.265354e+05,1.647472e+05,0.0,44.764983,1.000000e+05,1.666667e+05,1.000000e+06
sload,45332.0,8.478722e+07,1.676691e+08,0.0,19721.440918,5.066666e+07,1.000000e+08,5.268000e+09
dload,45332.0,2.405574e+04,1.358694e+05,0.0,0.000000,0.000000e+00,3.664603e+03,2.317180e+06
sinpkt,45332.0,8.083333e+01,8.008381e+02,0.0,0.006000,1.000000e-02,4.109095e+01,4.472990e+04
dinpkt,45332.0,7.781808e+01,7.963405e+02,0.0,0.000000,0.000000e+00,4.311379e+01,4.885760e+04


In [7]:
X = dataset[LIVE_FEATURES]
y = dataset["label"]
print(X.shape)
print(X.columns)
print(y.value_counts())

(82332, 13)
Index(['dur', 'proto', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sload',
       'dload', 'sinpkt', 'dinpkt', 'smean', 'dmean'],
      dtype='str')
label
1    45332
0    37000
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts())
print(y_test.value_counts())

(65865, 13)
(16467, 13)
label
1    36265
0    29600
Name: count, dtype: int64
label
1    9067
0    7400
Name: count, dtype: int64


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "proto_encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            ["proto"]
        )
    ],
    remainder="passthrough"
)

In [11]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)
print(X_train_processed.shape)
print(X_test_processed.shape)

(65865, 143)
(16467, 143)


In [12]:
from sklearn.ensemble import RandomForestClassifier

rf_live = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_live.fit(
    X_train_processed,
    y_train
)
y_pred = rf_live.predict(
    X_test_processed
)

In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9336248254083925
Precision: 0.9555530164533821
Recall: 0.9223557957428036
F1 Score: 0.9386609798529659

Confusion Matrix:
[[7011  389]
 [ 704 8363]]


In [14]:
import os
import joblib

LIVE_MODEL_DIR = "../models/live"

os.makedirs(LIVE_MODEL_DIR, exist_ok=True)

joblib.dump(
    rf_live,
    os.path.join(LIVE_MODEL_DIR, "live_random_forest.pkl")
)

joblib.dump(
    preprocessor,
    os.path.join(LIVE_MODEL_DIR, "live_preprocessor.pkl")
)

['../models/live\\live_preprocessor.pkl']

In [15]:
joblib.dump(
    LIVE_FEATURES,
    os.path.join(LIVE_MODEL_DIR, "live_features.pkl")
)

['../models/live\\live_features.pkl']